In [1]:
import torch
from torchvision.datasets import CIFAR10
from torchvision import transforms, models
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Subset, ConcatDataset


In [2]:
from torch.utils.data import Dataset
class AdvDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __getitem__(self, index):
        return self.x[index].permute(2,0,1).cpu(), self.y[index].item()

    def __len__(self):
        return len(self.x)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:

transform_cifar = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

transform_test = transforms.Compose([
    transforms.ToTensor()
])

In [5]:
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
train_dataset = Subset(train_dataset, range(30000))
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)
# test_dataset = Subset(test_dataset, range(5000)).dataset

# print(f'CIFAR10: train {train_dataset.data.shape}, test {test_dataset.data.shape}')

train_dataset_adv = torch.load('./adv_data/densenet_fgsm_cifar_train')
train_dataset_adv = Subset(train_dataset_adv, range(30000))
# test_dataset_adv = torch.load('./adv_data/resnet_pgd_cifar_test')
# test_dataset_adv = Subset(test_dataset_adv, range(5000)).dataset
# 
train_dataset = ConcatDataset([train_dataset, train_dataset_adv])
# test_dataset = ConcatDataset([test_dataset, test_dataset_adv])


Files already downloaded and verified
Files already downloaded and verified


In [6]:
batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [7]:
model = models.densenet121(weights="DEFAULT")

In [8]:
model

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [9]:
# modify the input and output layers
# model.features.conv0 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
model.classifier = nn.Linear(1024, 10)
model

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu0): ReLU(inplace=True)
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): _DenseBlock(
      (denselayer1): _DenseLayer(
        (norm1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu1): ReLU(inplace=True)
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu2): ReLU(inplace=True)
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
      (denselayer2): _DenseLayer(
        (norm1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu

In [10]:
lr = 0.001

model = model.to(device)
criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
best_val_loss = float('inf')
epochs = 10

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    tarin_bar = tqdm(train_loader, position=0, leave=True)
    for x, y in tarin_bar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        train_correct += class_pred.eq(y).sum().item()
    train_accuracy = train_correct / len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_bar = tqdm(test_loader, position=0, leave=True)
    with torch.no_grad():
        for x, y in val_bar:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = criterion(y_pred, y)
            val_loss += loss.item()
            class_pred = y_pred.argmax(dim=1)
            val_correct += class_pred.eq(y).sum().item()
    val_accuracy = val_correct / len(test_dataset)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), './model/DenseNet121_CIFAR_fgsm.pth')
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.6f}, Train Acc: {train_accuracy:.6f}, Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')
  

100%|██████████| 157/157 [00:02<00:00, 62.52it/s]


Epoch 1/10, Train Loss: 1.192005, Train Acc: 0.579967, Val Loss: 0.832941, Val Acc: 0.709200


100%|██████████| 157/157 [00:02<00:00, 63.77it/s]


Epoch 2/10, Train Loss: 1.149104, Train Acc: 0.601317, Val Loss: 0.882877, Val Acc: 0.699800


100%|██████████| 157/157 [00:02<00:00, 64.07it/s]


...

Epoch 8/10, Train Loss: 0.487282, Train Acc: 0.831100, Val Loss: 0.682195, Val Acc: 0.774100


100%|██████████| 157/157 [00:02<00:00, 66.94it/s]


Epoch 9/10, Train Loss: 0.433738, Train Acc: 0.849750, Val Loss: 0.719437, Val Acc: 0.770000


100%|██████████| 157/157 [00:02<00:00, 65.32it/s]

Epoch 10/10, Train Loss: 0.389353, Train Acc: 0.863783, Val Loss: 0.689333, Val Acc: 0.781700


In [11]:
model.load_state_dict(torch.load('model/DenseNet121_CIFAR_fgsm.pth'))
model = model.to(device)

In [12]:
model.eval()
val_loss = 0
val_correct = 0
val_bar = tqdm(test_loader, position=0, leave=True)
with torch.no_grad():
    for x, y in val_bar:
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = criterion(y_pred, y)
        val_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        val_correct += class_pred.eq(y).sum().item()
val_accuracy = val_correct / len(test_dataset)

print(f'Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')

100%|██████████| 157/157 [00:02<00:00, 63.60it/s]

Val Loss: 0.678310, Val Acc: 0.762900


In [13]:

test_dataset_adv = torch.load('./adv_data/densenet_fgsm_cifar_test')

test_loader_adv = DataLoader(test_dataset_adv, batch_size=batch_size, shuffle=False)

In [14]:
model.eval()
val_loss = 0
val_correct = 0
val_bar = tqdm(test_loader_adv, position=0, leave=True)
with torch.no_grad():
    for x, y in val_bar:
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = criterion(y_pred, y)
        val_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        val_correct += class_pred.eq(y).sum().item()
val_accuracy = val_correct / len(test_dataset)

print(f'Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')

100%|██████████| 157/157 [00:02<00:00, 71.53it/s]

Val Loss: 0.673715, Val Acc: 0.774300
